# 📅 Alerta Mensal de Evasão — Graduação Presencial
### a instituicao | v3 — cinco modelos, um por mês do semestre

O modelo consolidado do presencial só pontua no **fim do semestre**, quando a
rematrícula já está acontecendo. Este notebook treina **cinco modelos**, um para
cada mês letivo, para que a equipe de retenção possa ligar **todo mês** com o dado
que existe até ali.

| Modelo | Corte | Roda em | Vê |
|---|---|---|---|
| Mês 1 | 31/08 | setembro | 1 mês de aula |
| Mês 2 | 30/09 | outubro | 2 meses |
| Mês 3 | 31/10 | novembro | 3 meses |
| Mês 4 | 30/11 | dezembro | 4 meses |
| Mês 5 | 31/12 | janeiro | semestre fechado |

---

## ⚠️ Revisão de 22/09/2026 — o filtro de elegibilidade

Este notebook foi refeito depois de encontrar um vazamento temporal no filtro
`ELEGIVEL` da query, e um bug de rótulo que ele escondia. **Os números mudaram
para melhor em todos os cinco cortes.** O tópico 2 explica em detalhe.

Resumo do que mudou:

| Mês | Topo 200 antes | Topo 200 depois |
|---|---|---|
| 1 | 100 | **119** |
| 2 | 152 | **170** |
| 3 | 161 | **178** |
| 4 | 179 | **192** |
| 5 | 183 | **195** |

*(CV out-of-fold, 5 dobras, 5 sementes — a tabela do tópico 9 usa semente única
e por isso difere um pouco)*

---

## Por que cinco modelos e não um

Testado nas duas arquiteturas (CV out-of-fold, 3 sementes):

| Mês | 5 modelos | 1 modelo empilhado |
|---|---|---|
| 1 | 98 | **101** |
| 2 | **152** | 134 |
| 3 | **161** | 155 |
| 4 | **178** | 171 |
| 5 | **183** | 170 |

*(evasores reais no topo 200 da fila — medido **antes** da correção do filtro,
na base antiga. A comparação entre arquiteturas continua válida porque as duas
usavam a mesma base; os valores absolutos é que subiram desde então.)*

O modelo único recebia `MESES_DECORRIDOS` como feature e treinava com as cinco
fotografias empilhadas — 5× mais linhas. Perdeu mesmo assim.

O motivo é que o **mesmo valor tem significado oposto em meses diferentes**. Um
aluno com 10 dias ativos no AVA é bom no mês 1 (de 30 dias possíveis) e péssimo
no mês 5 (de 120). Empilhados, a árvore gasta capacidade aprendendo essa inversão
para cada feature, em vez de aprender o padrão de evasão.

---

## ⚠️ Este modelo vale para SEGUNDO semestre

Treinado na coorte **2025/2**, a única com log de AVA e alvo fechado. E primeiro
e segundo semestre têm populações diferentes:

| Semestre | % de calouros | Evasão |
|---|---|---|
| 1º (2024/1, 2025/1) | 29–34% | 15–18% |
| **2º (2024/2, 2025/2)** | **13–15%** | **24%** |

Aplicar em **2026/2** é seguro — mesmo tipo de semestre. Aplicar num primeiro
semestre (2027/1) não foi validado.

Em **janeiro de 2027**, quando 2026/2 fechar, o par `2026/1 → 2026/2` vira a
primeira coorte de 1º semestre com AVA. Aí dá para treinar a variante.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    recall_score,
    precision_score,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

LIMIAR = 0.30       # travado por decisao do projeto
FILA = 200          # ligacoes por mes
SEMENTE = 42
COORTE = "2025/2"

CORTES = [(1, "31/08"), (2, "30/09"), (3, "31/10"), (4, "30/11"), (5, "31/12")]

## 1. Carregamento dos Dados

Cinco bases, uma por corte, todas da **mesma coorte 2025/2**. O que muda entre
elas é só a data em que a fotografia foi tirada (`DT_CORTE_MANUAL` na query).

A população cresce com o corte: no mês 1, 368 alunos ainda não têm log no AVA e
ficam de fora (o `INNER JOIN` da query os exclui). No mês 5, sobram 10.

In [ ]:
DTYPE = {"ALUNO": str, "CURSO": str, "END_MUNICIPIO": str}

def carregar(mes):
    for pasta in ("../../tabelas", "tabelas"):
        for ext, ler in ((".csv", lambda p: pd.read_csv(p, encoding="utf-8-sig", dtype=DTYPE)),
                         (".xlsx", lambda p: pd.read_excel(p, dtype=DTYPE))):
            p = f"{pasta}/extracao_presencial_2025_2_m{mes}{ext}"
            if os.path.exists(p):
                return ler(p)
    raise FileNotFoundError(
        f"extracao_presencial_2025_2_m{mes} nao encontrado em tabelas/. "
        "A base e gitignored (dado pessoal) -- rode querys/01_extracao_presencial.sql "
        "com DT_CORTE_MANUAL no corte do mes. Parametros em querys/README.md."
    )

bases = {}
linhas = []
for mes, data in CORTES:
    d = carregar(mes)
    bases[mes] = d
    e = d[d["ELEGIVEL"] == 1]
    linhas.append({"Mês": mes, "Corte": f"{data}/2025", "Linhas": len(d),
                   "Elegíveis": len(e), "Evasores": int(e["EVADIU"].sum()),
                   "Taxa": f"{e['EVADIU'].mean():.2%}"})

print(f"Coorte {COORTE} em {len(CORTES)} cortes\n")
display(pd.DataFrame(linhas))

## 2. Elegibilidade — por que o filtro foi desativado (22/09/2026)

O filtro `ELEGIVEL == 1` descartava ~15% da base: quem havia concluído,
transferido ou cancelado. **Ele estava usando informação do fim do semestre para
montar a base de agosto.**

A fonte era `VW_ALUNO_SITUACAO_PERIODO`, que guarda **um estado por período, sem
data** — a view tem 4 colunas (`ALUNO`, `ANO`, `PERIODO`, `SITUACAO`). Não há
como fatiar por `DT_CORTE`. O resultado é que o filtro devolvia praticamente o
mesmo conjunto nos cinco cortes:

| Corte | Descartados | % |
|---|---|---|
| 31/08 | 1.289 | 14,9% |
| 30/09 | 1.355 | 15,1% |
| 31/10 | 1.365 | 15,1% |
| 30/11 | 1.372 | 15,1% |
| 31/12 | 1.372 | 15,1% |

Se respeitasse a data, esse número **cresceria** ao longo do semestre. Ficar
constante é a assinatura do vazamento.

### Não era métrica inflada — era treino que não batia com produção

Rodando em outubro na turma corrente, os ~1.100 alunos que vão concluir em
dezembro **estão na base**: ninguém sabe ainda que vão concluir. O modelo nunca
tinha visto esse perfil no treino.

### O bug de rótulo que o filtro escondia

Ao tirar o filtro, apareceram **834 formandos marcados como `EVADIU = 1`**. A
regra do `EVADIU` testava só a situação do período **alvo**, e o formando some
de lá — colou grau, não tem registro em 2026/1. Então o teste passava.

Corrigido na query: o `EVADIU` agora testa também a situação do período
observado. Tirar o filtro **sem** essa correção era desastre — 140 das 200
ligações iriam para formandos.

### O que foi medido

CV out-of-fold, 5 dobras, 5 sementes, evasores reais no topo 200 da fila:

| Mês | Corte | Com filtro | Sem filtro + rótulo corrigido | Ganho |
|---|---|---|---|---|
| 1 | 31/08 | 100,4 | **119,0** | +18,6 |
| 2 | 30/09 | 151,8 | **170,4** | +18,6 |
| 3 | 31/10 | 161,4 | **178,2** | +16,8 |
| 4 | 30/11 | 179,4 | **191,6** | +12,2 |
| 5 | 31/12 | 183,4 | **194,6** | +11,2 |

Os cinco melhoram, e o ganho é **maior nos cortes precoces** — justamente onde a
distância entre a fotografia do corte e a informação de dezembro era maior. O
mês 1, que é o alerta mais valioso e era o mais fraco, ganha 19%.

Formandos que caem no topo 200: **0 a 0,2 por corte**. O modelo separa formando
de evasor sozinho, por crédito acumulado e disciplinas cursadas. A feature
`PROGRESSO_CURSO`, cogitada para resolver isso, foi medida como desnecessária.

### Tentativa descartada

Excluir quem pediu transferência até `DT_CORTE`, via `ERP_SOLICITACAO_SERV.DATA`
— um dado que **tem** data e respeitaria o corte. Medido: derruba o topo 200 de
170,4 para **147,8**, pior que o filtro antigo. Excluía 317 alunos, dos quais
141 eram evasores reais. E o aluno que pede transferência em setembro continua
matriculado em setembro: é exatamente quem a retenção deveria ligar.

> `ELEGIVEL` agora é constante 1 na query. A linha de filtro abaixo fica por
> compatibilidade e é no-op. A auditoria de quem concluiu está em
> `AUDITORIA_STATUS` — nada se perdeu.


In [ ]:
# ELEGIVEL virou constante 1 na query (ver o texto acima). O filtro fica por
# compatibilidade e e no-op -- 'descartados' tem que sair 0 nos cinco cortes.
# Quem ja encerrou o curso ate o corte sai da base (revisao de 23/09/2026).
# Mesma regra do bloco 'encerrado' de querys/09_job_presencial_diario.sql:
# treino e producao precisam ver a mesma populacao. O CSV sai de
# querys/11_encerramentos_presencial.sql.
def carregar_encerramentos():
    for pasta in ("../../tabelas", "tabelas"):
        p = f"{pasta}/encerramentos_presencial_2025_2.csv"
        if os.path.exists(p):
            return pd.read_csv(p, encoding="utf-8-sig", dtype={"ALUNO": str},
                               parse_dates=["DT_ENCERRAMENTO"])
    raise FileNotFoundError(
        "encerramentos_presencial_2025_2.csv nao encontrado em tabelas/ -- "
        "rode querys/11_encerramentos_presencial.sql e salve o resultado la.")

encerr = carregar_encerramentos().set_index("ALUNO")["DT_ENCERRAMENTO"]
DATA_CORTE = {m: pd.Timestamp(f"2025-{d[3:5]}-{d[0:2]}") for m, d in CORTES}

elegiveis, ja_sairam = {}, {}
for mes, data in CORTES:
    d = bases[mes]
    d = d[d["ELEGIVEL"] == 1]
    saiu = d["ALUNO"].astype(str).map(encerr) <= DATA_CORTE[mes]
    ja_sairam[mes] = d[saiu]
    elegiveis[mes] = d[~saiu].copy()
    desc = len(bases[mes]) - len(elegiveis[mes])
    conc = int((elegiveis[mes]["AUDITORIA_STATUS"] == "CONCLUIU").sum())
    print(f"Mês {mes} ({data}): {len(elegiveis[mes]):,} alunos  "
          f"({desc:,} ja tinham encerrado o curso, "
          f"{int(ja_sairam[mes]['EVADIU'].sum())} deles evasores)  |  "
          f"{int(elegiveis[mes]['EVADIU'].sum())} evasores  |  "
          f"{conc:,} concluintes na base")

print(f"\nAlunos que só passam a existir depois do mês 1: "
      f"{len(elegiveis[5]) - len(elegiveis[1])} "
      "(ainda não tinham log no AVA em agosto)")

# Checagem do rotulo corrigido: concluinte NUNCA pode estar marcado como evasor.
# Antes da correcao de 22/09/2026 eram 834 no corte de setembro.
for mes, _ in CORTES:
    e = elegiveis[mes]
    falsos = int(e[e["AUDITORIA_STATUS"] == "CONCLUIU"]["EVADIU"].sum())
    assert falsos == 0, f"mes {mes}: {falsos} concluintes rotulados como evasores"
print("\n✅ Rótulo conferido: nenhum concluinte marcado como evasor nos 5 cortes.")


## 3. Engenharia de Features de Risco Combinado

Aplicada a cada corte separadamente.

Uma decisão medida: a sugestão de **remover contadores absolutos** (`FALTA_TOTAL`,
`HIST_DISC_*`) nos cortes precoces, mantendo só taxas, **não pagou** — no mês 1
piorou (96 contra 100 evasores no topo 200), no mês 2 ficou idêntico. As features
são as mesmas nos cinco.


In [ ]:
def engenharia(d):
    d = d.copy()
    d["EST_CIVIL"] = (d["EST_CIVIL"].astype(str).str.upper()
                      .replace({"NAN": "NAO_INFORMADO", "NONE": "NAO_INFORMADO"}))
    vc = d["END_MUNICIPIO"].value_counts()
    principais = vc[vc >= 30].index.tolist()
    d["END_MUNICIPIO_AGRUPADO"] = d["END_MUNICIPIO"].apply(
        lambda x: str(x) if x in principais else "OUTRO")
    d["RATIO_GAP_ATIVOS"] = d["AVA_MAIOR_GAP_DIAS"] / (d["AVA_DIAS_ATIVOS"] + 1)
    d["SESSOES_POR_DIA"] = d["AVA_QTD_SESSOES"] / (d["AVA_DIAS_ATIVOS"] + 1)
    return d, principais

feat, municipios = {}, {}
for mes, _ in CORTES:
    feat[mes], municipios[mes] = engenharia(elegiveis[mes])
print("✅ Engenharia aplicada aos 5 cortes.")
print(f"Municípios mantidos por corte: "
      f"{ {m: len(municipios[m]) for m, _ in CORTES} }")

## 4. Separação de $X$ (Features) e $y$ (Target)

A lista de exclusão herda o corte de ruído medido em 18/09/2026 no modelo
consolidado — 10 blocos, de 61 para 41 colunas, sem perda de desempenho.

> Aquele modelo (`ml/grad-presencial/`) foi **removido em 23/09/2026**: era o
> corte de 31/12, idêntico ao mês 5 daqui, mas com a base anterior à correção do
> rótulo e uma suíte de testes que media memória. A medição do corte de ruído
> continua válida — a base mudou depois, mas as colunas cortadas eram ruído em
> qualquer versão dela.

`SEXO` fica fora do `X` por decisão ética do `CLAUDE.md` — continua na base para
auditoria de viés.


In [ ]:
colunas_remover = [
    # identificacao, alvo e metadados
    "ALUNO", "CURSOU_ALVO", "SIT_ALVO", "ELEGIVEL", "SIT_FIM_OBS",
    "AUDITORIA_STATUS", "DT_INICIO_OBS", "DT_CORTE", "ANO_OBS", "PERIODO_OBS",
    "MES_CORTE_N", "NOME_CURSO", "EVADIU",
    # quase constantes
    "NACIONALIDADE", "UNIDADE_FISICA",
    # substituidas por versoes tratadas
    "END_MUNICIPIO",
    # redundantes / colineares
    "FREQ_MEDIA", "TOT_AULAS_PREVISTAS", "AVA_QTD_SESSOES", "AVA_DIAS_ATIVOS",
    "AVA_SESSOES_RECENTES", "AVA_DIAS_ATIVOS_RECENTES",
    # corte de ruido medido no modelo consolidado
    "EST_CIVIL", "SEXO", "MODALIDADE_CURSO", "TURNO", "PER_INGRESSO",
    "DISC_REP_NOTA_ANT", "DISC_REP_FREQ_ANT", "FREQ_PCT_DISC_REPROVA",
    "RISCO_NOTA_BAIXA",
    # ano absoluto: o modelo cortava em 2024,5 / 2025,5 e, aplicado em 2026/2,
    # jogaria os 3.555 ingressantes de 2026 numa faixa que no treino tinha 187
    # alunos. ANOS_DE_CASA carrega a mesma informacao e vale em qualquer
    # semestre (revisao de 23/09/2026).
    "ANO_INGRESSO",
]

X, y = {}, {}
for mes, _ in CORTES:
    d = feat[mes]
    y[mes] = d["EVADIU"].astype(int).values
    X[mes] = d.drop(columns=[c for c in colunas_remover if c in d.columns])

print(f"Features em X: {X[1].shape[1]}\n")
print(X[1].columns.tolist())

## 5. Encoding de Categóricas e Imputação pela Mediana

O `SimpleImputer` do processo padrão é substituído por `fillna(mediana do treino)`
dentro de cada dobra da validação cruzada — calcular a mediana sobre a base toda
vazaria informação do conjunto de avaliação para o de treino.

In [ ]:
X_enc = {}
for mes, _ in CORTES:
    cat = X[mes].select_dtypes(include=["object", "category", "string"]).columns.tolist()
    e = pd.get_dummies(X[mes], columns=cat, drop_first=True)
    X_enc[mes] = e.drop(columns=[c for c in e.columns if e[c].isna().all()])

print(f"Categóricas: {cat}\n")
for mes, data in CORTES:
    n = X_enc[mes].isna().sum()
    print(f"Mês {mes} ({data}): {X_enc[mes].shape[1]} colunas | "
          f"{int((n > 0).sum())} com nulo | "
          f"maior lacuna: {100*n.max()/len(X_enc[mes]):.0f}%")

## 6. Validação Cruzada Out-of-Fold (em vez de split 80/20)

Aqui o notebook se afasta do processo padrão de propósito, e a razão é medida.

O split 80/20 avalia **1.816 alunos**. Tirar o topo 200 *dessa fatia* é pegar os
**11% melhores**; a fila real da equipe é o topo 200 de **9.079**, ou seja, os
**2,2% melhores**. Escolher os 11% é muito mais fácil.

Medido em 23/09/2026 sobre a base corrigida, 5 sementes:

| Método | `recall@200` | Erro |
|---|---|---|
| Topo 200 da fatia de teste | 59,0% ±2,1 | **+41,2 pp** |
| Split, mas com K proporcional | 17,7% ±0,7 | +0,1 pp |
| **CV out-of-fold nos 9.079** | **17,8% ±0,1** | — |

Duas leituras importam aqui.

**A armadilha continua viva.** A linha de cima é o que este notebook reportaria
se alguém tirasse o ajuste de K proporcional — 41 pontos acima do valor real. O
CV não conserta um erro: ele **remove a possibilidade** de cometê-lo.

**O ganho do CV não é número melhor, é estabilidade e cobertura.** Contra o split
com K proporcional o resultado empata (17,7% contra 17,8%), mas o desvio-padrão
cai de ±0,7 para ±0,1, e **todo aluno recebe nota** — o que permite montar a fila
real, rodar os cadernos de teste sem medir memória e calcular a política de não
repetir contato do tópico 10.


In [ ]:
def treinar_e_pontuar(Xm, ym, semente=SEMENTE, folds=5):
    # Devolve a probabilidade out-of-fold de cada aluno e o modelo de producao.
    oof = np.zeros(len(ym))
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=semente)
    for i_tr, i_te in skf.split(Xm, ym):
        med = Xm.iloc[i_tr].median()          # mediana SO do treino da dobra
        m = RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_split=4,
            min_samples_leaf=2, class_weight={0: 1.0, 1: 12.0},
            random_state=semente, n_jobs=-1)
        m.fit(Xm.iloc[i_tr].fillna(med), ym[i_tr])
        oof[i_te] = m.predict_proba(Xm.iloc[i_te].fillna(med))[:, 1]
    # modelo de producao: treinado com tudo
    med_full = Xm.median()
    final = RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=4,
        min_samples_leaf=2, class_weight={0: 1.0, 1: 12.0},
        random_state=semente, n_jobs=-1).fit(Xm.fillna(med_full), ym)
    return oof, final, med_full


proba, modelos, medianas = {}, {}, {}
for mes, data in CORTES:
    proba[mes], modelos[mes], medianas[mes] = treinar_e_pontuar(X_enc[mes], y[mes])
    print(f"✅ Mês {mes} ({data}): {len(y[mes]):,} alunos pontuados out-of-fold")

## 7. Treinamento dos Cinco Modelos (Random Forest Sensível a Custo)

`class_weight={0: 1.0, 1: 12.0}` nos cinco, para manter a comparação justa entre
os cortes. O peso vem da medição feita no presencial consolidado, quando ele
ainda existia.


In [ ]:
print(f"{'Mês':>4s} {'Corte':>7s} {'Alunos':>8s} {'Evasores':>9s} "
      f"{'Árvores':>8s} {'Colunas':>8s} {'Peso classe':>12s}")
print("-" * 64)
for mes, data in CORTES:
    m = modelos[mes]
    print(f"{mes:4d} {data:>7s} {len(y[mes]):8,} {int(y[mes].sum()):9d} "
          f"{m.n_estimators:8d} {X_enc[mes].shape[1]:8d} "
          f"{'1:12':>12s}")

## 8. O Limiar de Decisão e o Tamanho da Fila

O limiar está travado em **0,30**. A tabela mostra o que cada corte custa — não
serve para escolher outro.

Repare que o **recall no limiar quase não distingue os meses** (todos entre 90% e
93%), porque nos cinco a lista passa de 4.000 nomes numa base de ~7.500. Com 60%
da escola marcada, alto recall é aritmética.

Quem separa os modelos é a **ordem da fila**, medida no tópico 9.

In [ ]:
linhas = []
for mes, data in CORTES:
    for t in (0.50, 0.30, 0.20):
        pred = (proba[mes] >= t).astype(int)
        linhas.append({
            "Mês": mes, "Corte": data, "Limiar": f"{t:.2f}",
            "Recall": f"{recall_score(y[mes], pred, zero_division=0):.1%}",
            "Precisão": f"{precision_score(y[mes], pred, zero_division=0):.1%}",
            "Alertas": int(pred.sum()),
            "% da base": f"{pred.sum()/len(y[mes]):.0%}",
        })
display(pd.DataFrame(linhas))

## 9. Avaliação: quantos evasores a fila alcança

A métrica que manda, na ordem do `CLAUDE.md`: `recall@200`, depois `recall@10%`,
depois PR-AUC. Acurácia não entra em nenhuma decisão.

In [ ]:
def metricas(ym, p):
    pred = (p >= LIMIAR).astype(int)
    o = np.argsort(-p)
    return {
        "recall@200": ym[o[:FILA]].sum() / ym.sum(),
        "topo200": int(ym[o[:FILA]].sum()),
        "prec@200": ym[o[:FILA]].sum() / FILA,
        "recall@10%": ym[o[:int(0.10 * len(ym))]].sum() / ym.sum(),
        "pr_auc": average_precision_score(ym, p),
        "roc_auc": roc_auc_score(ym, p),
        "recall_limiar": recall_score(ym, pred, zero_division=0),
        "prec_limiar": precision_score(ym, pred, zero_division=0),
        "alertas": int(pred.sum()),
    }

res = {mes: metricas(y[mes], proba[mes]) for mes, _ in CORTES}
tab = pd.DataFrame([{
    "Mês": mes, "Corte": data, "Evasores": int(y[mes].sum()),
    "Topo 200 pega": res[mes]["topo200"],
    "recall@200": f"{res[mes]['recall@200']:.1%}",
    "prec@200": f"{res[mes]['prec@200']:.1%}",
    "recall@10%": f"{res[mes]['recall@10%']:.1%}",
    "PR-AUC": f"{res[mes]['pr_auc']:.3f}",
    "ROC-AUC": f"{res[mes]['roc_auc']:.3f}",
    "Recall @0,30": f"{res[mes]['recall_limiar']:.1%}",
    "Alertas": res[mes]["alertas"],
} for mes, data in CORTES])
display(tab)

print("\nLeitura: esperar o semestre avançar compra precisão. Do mês 1 ao mês 5,")
print(f"o topo {FILA} sai de {res[1]['topo200']} para {res[5]['topo200']} evasores")
print(f"({res[1]['prec@200']:.0%} -> {res[5]['prec@200']:.0%} de acerto por ligação).")

## 10. A Política de Fila: por que não repetir importa mais que o modelo

Com ligação mensal, a pergunta deixa de ser "quantos evasores pego neste mês" e
passa a ser **"quantos evasores distintos alcanço no semestre"**.

Duas políticas, mesmas 1.000 ligações:

- **Ingênua** — todo mês pega o topo 200 daquele mês. Quem está em risco em
  setembro continua em risco em outubro, então a lista se repete.
- **Sem repetir** — todo mês pega os 200 melhores **que ainda não foram ligados**.

In [ ]:
por_mes = {mes: pd.DataFrame({"aluno": feat[mes]["ALUNO"].astype(str).values,
                              "p": proba[mes], "evadiu": y[mes]})
           for mes, _ in CORTES}
TOTAL_EVASORES = int(y[5].sum())

resultados = {}
for politica in ("Ingênua", "Sem repetir"):
    ligados, alcancados, hist = set(), set(), []
    for mes, data in CORTES:
        d = por_mes[mes].sort_values("p", ascending=False)
        if politica == "Sem repetir":
            d = d[~d["aluno"].isin(ligados)]
        fila = d.head(FILA)
        repetidos = len(set(fila["aluno"]) & ligados)
        alcancados |= set(fila.loc[fila["evadiu"] == 1, "aluno"])
        ligados |= set(fila["aluno"])
        hist.append({"Política": politica, "Mês": mes, "Corte": data,
                     "Evasores na fila": int(fila["evadiu"].sum()),
                     "Já ligados antes": repetidos,
                     "Evasores alcançados": len(alcancados),
                     "% da evasão": f"{len(alcancados)/TOTAL_EVASORES:.1%}"})
    resultados[politica] = (hist, len(ligados), len(alcancados))

display(pd.DataFrame([h for p in resultados for h in resultados[p][0]]))

print(f"\nAo fim dos 5 meses, com {5*FILA:,} ligações em ambos os casos:")
for p, (_, n_lig, n_alc) in resultados.items():
    print(f"  {p:12s}: {n_lig:5,} alunos distintos | "
          f"{n_alc:3d} evasores alcançados de {TOTAL_EVASORES} "
          f"({n_alc/TOTAL_EVASORES:.1%})")

## 11. Curvas: captura acumulada e Precision-Recall por mês

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# a) captura acumulada nas duas politicas
for pol, cor in [("Ingênua", "#A0AEC0"), ("Sem repetir", "#1A365D")]:
    h = resultados[pol][0]
    axes[0].plot([x["Mês"] for x in h], [x["Evasores alcançados"] for x in h],
                 "o-", lw=2.6, ms=7, color=cor, label=pol)
axes[0].set_title("Evasores alcançados ao longo do semestre\n"
                  f"({FILA} ligações por mês nos dois casos)", fontweight="bold")
axes[0].set_xlabel("Mês do semestre", fontweight="bold")
axes[0].set_ylabel("Evasores distintos alcançados", fontweight="bold")
axes[0].set_xticks([m for m, _ in CORTES])
axes[0].legend()
axes[0].grid(alpha=0.3)

# b) precision-recall por mes
cores = plt.cm.viridis(np.linspace(0.15, 0.85, len(CORTES)))
for (mes, data), cor in zip(CORTES, cores):
    pr, rc, _ = precision_recall_curve(y[mes], proba[mes])
    axes[1].plot(rc, pr, lw=2.2, color=cor,
                 label=f"Mês {mes} ({data}) — PR-AUC {res[mes]['pr_auc']:.3f}")
axes[1].axhline(y[5].mean(), color="crimson", ls="--", lw=1.4,
                label=f"Baseline ({y[5].mean():.1%})")
axes[1].set_title("Curva Precision-Recall por corte", fontweight="bold")
axes[1].set_xlabel("Recall", fontweight="bold")
axes[1].set_ylabel("Precisão", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Importância das Features — como o sinal muda ao longo do semestre

In [ ]:
imp = pd.DataFrame({f"Mês {mes}": pd.Series(modelos[mes].feature_importances_,
                                            index=X_enc[mes].columns)
                    for mes, _ in CORTES}).fillna(0)
top = imp["Mês 5"].sort_values(ascending=False).head(12).index

fig, ax = plt.subplots(figsize=(11, 6))
for f in top:
    ax.plot([m for m, _ in CORTES], [100 * imp.loc[f, f"Mês {m}"] for m, _ in CORTES],
            "o-", lw=2, ms=5, label=f)
ax.set_title("Importância das features ao longo do semestre", fontweight="bold", pad=12)
ax.set_xlabel("Mês do corte", fontweight="bold")
ax.set_ylabel("Importância relativa (%)", fontweight="bold")
ax.set_xticks([m for m, _ in CORTES])
ax.legend(fontsize=8, ncol=2, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 8 de cada corte:\n")
display(pd.DataFrame({f"Mês {m}": imp[f"Mês {m}"].sort_values(ascending=False)
                      .head(8).index.tolist() for m, _ in CORTES}))

## 13. Salvamento do Artefato (`.pkl`)

Um arquivo com os cinco modelos, as medianas de imputação de cada corte, e a
política de fila. A função `montar_fila` implementa a regra de não repetir.

In [ ]:
pkl_ml = "modelo_evasao_presencial_mensal.pkl"
pkl_testes = os.path.join("..", "..", "area-testes", "presencial-mensal",
                          "modelo_evasao_presencial_mensal.pkl")

VERSAO_MODELO = "presencial_mensal_2025_2_v2"

# Regra do CLAUDE.md para artefato: gravar o que o predict aceita e conferir.
for mes, _ in CORTES:
    assert modelos[mes].n_features_in_ == X_enc[mes].shape[1], f"mes {mes}"
    assert list(modelos[mes].feature_names_in_) == X_enc[mes].columns.tolist(), f"mes {mes}"

artifact = {
    "versao": VERSAO_MODELO,
    "populacao": "sem quem encerrou o curso ate DT_CORTE (ERP_H_CURSOS_CONCL)",
    "colunas_modelo": {m: X_enc[m].columns.tolist() for m, _ in CORTES},
    "features_pre_encoding": {m: X[m].columns.tolist() for m, _ in CORTES},
    "modelos": modelos,
    "medianas": medianas,
    "feature_names": {m: X_enc[m].columns.tolist() for m, _ in CORTES},
    "principais_municipios": municipios,
    "colunas_remover": colunas_remover,
    "threshold": LIMIAR,
    "fila": FILA,
    "cortes": {m: f"{d}/2025" for m, d in CORTES},
    "coorte_treino": COORTE,
    "tipo_semestre": "2o semestre (ago-dez)",
    "politica_fila": "sem repetir -- excluir alunos ja contatados nos meses anteriores",
    # Probabilidade OUT-OF-FOLD de cada aluno, por corte. Serve para os
    # cadernos de area-testes/: a coorte 2025/2 e a unica que existe, entao
    # pontuar com o modelo final mediria memoria. Com a out-of-fold, cada
    # aluno tem nota de um modelo que nao o viu.
    "proba_oof": {m: dict(zip(feat[m]["ALUNO"].astype(str), proba[m]))
                  for m, _ in CORTES},
    "metadata": {
        "modalidade": "Graduação Presencial — alerta mensal",
        "validacao": "CV out-of-fold 5 dobras (nao holdout: ver topico 6)",
        "por_mes": {m: {k: float(v) if isinstance(v, (int, float, np.floating)) else v
                        for k, v in res[m].items()} for m, _ in CORTES},
        "evasores_alcancados_ingenua": resultados["Ingênua"][2],
        "evasores_alcancados_sem_repetir": resultados["Sem repetir"][2],
        "total_evasores": TOTAL_EVASORES,
    },
}

joblib.dump(artifact, pkl_ml, compress=3)
print(f"🎉 Salvo em: {pkl_ml}  ({os.path.getsize(pkl_ml)/(1024*1024):.1f} MB)")

destino = os.path.dirname(pkl_testes)
if os.path.isdir(destino):
    joblib.dump(artifact, pkl_testes, compress=3)
    print(f"🎉 Copiado para: {pkl_testes}")
else:
    print(f"⚠️  AVISO: {destino} nao existe -- copia NAO feita.")

print("\nResumo gravado:")
for mes, data in CORTES:
    print(f"  Mês {mes} ({data}): topo {FILA} pega {res[mes]['topo200']} evasores "
          f"| PR-AUC {res[mes]['pr_auc']:.3f}")

## 14. Matriz de Correlação de Pearson

Comparação entre o primeiro e o último corte — mostra quais variáveis já carregam
sinal no mês 1 e quais só aparecem quando o semestre avança.

O PNG oficial vai para
`relatorios/matriz-correlacao/matriz_correlacao_presencial_mensal.png`.

In [ ]:
# Um heatmap em tamanho cheio (corte final, a referencia) -- dois empilhados
# num quadro de 1650x1350 ficam ilegiveis. A comparacao entre cortes vira o
# ranking da celula seguinte, que e onde a pergunta realmente esta.
num = [c for c in X[5].select_dtypes(include=[np.number]).columns]
dc = feat[5][[c for c in num if c in feat[5].columns]].copy()
dc.insert(0, "EVADIU", y[5])
corr = dc.corr()

plt.figure(figsize=(11, 9))   # 11x9in @150dpi = 1650x1350 px
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), cmap="coolwarm",
            vmax=0.5, vmin=-0.5, center=0, square=True, linewidths=0.5,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            cbar_kws={"shrink": 0.8})
plt.title(f"Matriz de Correlação de Pearson — Presencial, corte final (31/12)\n"
          f"({len(dc):,} alunos elegíveis, {int(y[5].sum())} evasores)",
          fontsize=12, fontweight="bold", pad=14)
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()

destino = os.path.join("..", "..", "relatorios", "matriz-correlacao")
if os.path.isdir(destino):
    saida = os.path.join(destino, "matriz_correlacao_presencial_mensal.png")
    plt.savefig(saida, dpi=150)   # sem bbox_inches: ele recorta e quebra o padrao
    print(f"PNG oficial salvo em: {saida}")
plt.show()

In [ ]:
# --- ranking: correlacao com EVADIU no primeiro vs no ultimo corte ----------
# E aqui que a comparacao entre cortes fica legivel -- um heatmap por corte,
# lado a lado, vira ruido visual.
evol = {}
for mes, _ in CORTES:
    d = feat[mes][[c for c in num if c in feat[mes].columns]].copy()
    d["EVADIU"] = y[mes]
    evol[mes] = d.corr()["EVADIU"].drop("EVADIU")
evol = pd.DataFrame(evol).dropna()
evol["ganho"] = evol[5].abs() - evol[1].abs()

ordem = evol[5].abs().sort_values(ascending=False).head(14).index[::-1]
pos = np.arange(len(ordem))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(pos + 0.20, evol.loc[ordem, 1], 0.38, color="#A0AEC0",
        edgecolor="black", lw=0.4, label="Mês 1 (31/08) — 1 mês de aula")
ax.barh(pos - 0.20, evol.loc[ordem, 5], 0.38, color="#1A365D",
        edgecolor="black", lw=0.4, label="Mês 5 (31/12) — semestre fechado")
ax.axvline(0, color="gray", ls="--", lw=0.9)
ax.set_yticks(pos)
ax.set_yticklabels(ordem, fontsize=8.5)
ax.set_xlabel("Correlação de Pearson com EVADIU   —   negativo retém, positivo arrisca",
              fontsize=10, fontweight="bold")
ax.set_title("Quanto cada variável informa sobre a evasão, no início e no fim do semestre",
             fontsize=12, fontweight="bold", pad=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("=" * 76)
print("AS QUE MAIS GANHAM SINAL COM O PASSAR DO SEMESTRE")
print("=" * 76)
tab = evol.reindex(evol["ganho"].sort_values(ascending=False).index).head(8)
display(tab[[1, 5, "ganho"]].round(3).rename(columns={1: "Mês 1", 5: "Mês 5"}))

print("\nAS QUE JA INFORMAM NO MES 1 (nao precisam do semestre inteiro)")
print("=" * 76)
ja = evol.reindex(evol[1].abs().sort_values(ascending=False).index).head(8)
display(ja[[1, 5, "ganho"]].round(3).rename(columns={1: "Mês 1", 5: "Mês 5"}))

# --- multicolinearidade no corte final --------------------------------------
pares = [{"Feature 1": a, "Feature 2": b, "r": round(corr.loc[a, b], 3)}
         for i, a in enumerate(corr.columns) for b in corr.columns[i+1:]
         if abs(corr.loc[a, b]) >= 0.70]
print("\n" + "=" * 76)
print("DIAGNÓSTICO DE MULTICOLINEARIDADE NO CORTE FINAL (|r| >= 0,70)")
print("=" * 76)
print(pd.DataFrame(pares).sort_values("r", key=abs, ascending=False).to_string(index=False)
      if pares else "Nenhum par com multicolinearidade severa.")

## 15. O Teste que Decide: o modelo ganha de uma regra de uma coluna?

Critério do `CLAUDE.md`, herdado da pós EAD:

> *"Só publicar modelo que supere a regra de um critério por margem que sobreviva
> ao intervalo de confiança."*

Lá esse teste **derrubou o modelo**: ordenar por `AVA_DIAS_ATIVOS`, sem treino
nenhum, acertava exatamente os mesmos evasores que vinte features.

O adversário é `AVA_DIAS_DESDE_ULT_ACESSO` — o sinal que o `CLAUDE.md` aponta
como o mais forte em todas as modalidades. Ordena do maior para o menor, pega os
200 primeiros, pronto. Sem modelo.

**Os dois competem sobre os mesmos alunos**, com a mesma métrica. O `CLAUDE.md`
registra que comparar o modelo out-of-fold com o acaso simulado sobre outra base
já inverteu uma conclusão neste projeto.

### Resultado (topo 200, probabilidade out-of-fold)

| Mês | Corte | Modelo | Regra de 1 coluna | Acaso | Diferença |
|---|---|---|---|---|---|
| 1 | 31/08 | **123** | 78 | 23 | **+45** |
| 2 | 30/09 | **168** | 136 | 24 | **+32** |
| 3 | 31/10 | **180** | 150 | 24 | **+30** |
| 4 | 30/11 | **192** | 162 | 24 | **+30** |
| 5 | 31/12 | **195** | 166 | 24 | **+29** |

**Os cinco passam com folga larga.** Para comparar: na Pós Live o marco de 90
dias passou por apenas +2,4 evasores, no limite do critério.

O padrão se repete aqui: **o modelo vale mais quando tem menos dado.** No mês 1 a
vantagem é de 45 evasores; no mês 5, de 29. Quando `dias sem acessar` já separa
bem — porque quem vai evadir já sumiu —, o modelo tem menos a acrescentar.

> A regra sozinha já bate o acaso com muita folga (78 contra 23 no mês 1). Se um
> dia for preciso operar sem `.pkl`, "ordene por dias sem acessar o AVA" é um
> ponto de partida honesto — mas aqui custaria de 29 a 45 evasores por ciclo.


In [ ]:
# Os dois competem sobre os MESMOS alunos, com a MESMA metrica.
REGRA_OBVIA = ("AVA_DIAS_DESDE_ULT_ACESSO", +1)   # +1 = maior valor, maior risco


def topo_k(ym, score, k=FILA):
    s = pd.Series(score).fillna(-np.inf).values
    return int(ym[np.argsort(-s)[:k]].sum())


linhas_regra = []
for mes, data in CORTES:
    d, ym = feat[mes], y[mes]
    N, EV = len(ym), int(ym.sum())

    mod = topo_k(ym, proba[mes])

    candidatas = [c for c in d.columns
                  if pd.api.types.is_numeric_dtype(d[c])
                  and c not in ("EVADIU", "CURSOU_ALVO", "ELEGIVEL",
                                "ANO_OBS", "PERIODO_OBS", "MES_CORTE_N")
                  and d[c].nunique(dropna=True) > 1]
    res = sorted(((c, sg, topo_k(ym, sg * d[c].astype(float)))
                  for c in candidatas for sg in (+1, -1)), key=lambda r: -r[2])
    obvia = next(r for r in res if r[0] == REGRA_OBVIA[0] and r[1] == REGRA_OBVIA[1])

    linhas_regra.append({
        "Mês": mes, "Corte": data,
        "Modelo (topo 200)": mod,
        "Regra 1 coluna": obvia[2],
        "Melhor regra": f"{res[0][2]} [{res[0][0]}]",
        "Acaso": round(FILA / N * EV),
        "Modelo - regra": mod - obvia[2],
    })

display(pd.DataFrame(linhas_regra))

print("\nAs 5 melhores regras de uma coluna, por corte:")
for mes, data in CORTES:
    d, ym = feat[mes], y[mes]
    candidatas = [c for c in d.columns
                  if pd.api.types.is_numeric_dtype(d[c])
                  and c not in ("EVADIU", "CURSOU_ALVO", "ELEGIVEL",
                                "ANO_OBS", "PERIODO_OBS", "MES_CORTE_N")
                  and d[c].nunique(dropna=True) > 1]
    res = sorted(((c, sg, topo_k(ym, sg * d[c].astype(float)))
                  for c in candidatas for sg in (+1, -1)), key=lambda r: -r[2])[:5]
    print(f"\n   Mês {mes} ({data}):")
    for c, sg, t in res:
        print(f"      {t:3d}   ordenar por {c} "
              f"({'maior' if sg > 0 else 'menor'} primeiro)")
